# Explainable Rehabilitation Motion Classification — quickstart

Runs the whole pipeline on Google Colab. You need the course recordings
(`기계학습_1차/` and the unlabelled `demo/` folder) on your Drive.

Runtime → Change runtime type → **CPU** is enough; the model has 63 k parameters.

In [ ]:
!git clone https://github.com/lmm0125-art/machine-learning.git
%cd machine-learning
!pip install -q -e .


### 1. Point the loader at the data on Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/ml1/dataset/기계학습_1차'
DEMO_DIR  = '/content/drive/MyDrive/ml1/dataset/demo'

### 2. Check that the parser sees what we expect

In [ ]:
!python scripts/check_data.py --root "$DATA_ROOT"

### 3. Train

5-fold group CV over recordings, three seeds. On a Colab CPU this takes a few
minutes; drop `--seeds` to 1 for a quick look.

In [ ]:
!python -m rehab_cls.train --data-root "$DATA_ROOT" --out-dir runs/colab \
    --cv 5 --seeds 3 --features raw_delta --per-recording-norm \
    --hidden 96 --dropout 0.4 --epochs 80 --patience 0 --train-final

### 4. Predict the unlabelled demo recordings

In [ ]:
!python -m rehab_cls.predict --checkpoint runs/colab/checkpoint_final.pt \
    --input "$DEMO_DIR" --out-dir runs/colab_predict --max-plots 0

import pandas as pd
pd.read_csv('runs/colab_predict/predictions_per_recording.csv')

### 5. Look at the attention of one repetition

In [ ]:
import glob
from IPython.display import Image, display

!python -m rehab_cls.predict --checkpoint runs/colab/checkpoint_final.pt \
    --input "$DEMO_DIR" --out-dir runs/colab_plots --max-plots 3

for path in sorted(glob.glob('runs/colab_plots/attention/*.png'))[:3]:
    print(path)
    display(Image(path))